In [2]:
import hand_tracker

tracker = hand_tracker.HandTracker()

landmarks = list(tracker.track())

I0000 00:00:1781603209.034501  226308 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
W0000 00:00:1781603209.040788  226311 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781603209.046435  226313 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781603210.719003  226315 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


In [7]:
opened = list(tracker.track())

In [9]:
closed = list(tracker.track())

In [13]:
opened = np.stack(opened)[:, 0]
closed = np.stack(closed)[:, 0]

In [20]:
import tensorflow as tf

ds_opened = tf.data.Dataset.from_tensor_slices(opened).map(
    lambda x: (tf.reshape(x, [-1]), 0))
ds_closed = tf.data.Dataset.from_tensor_slices(closed).map(
    lambda x: (tf.reshape(x, [-1]), 1))

ds = tf.data.Dataset.sample_from_datasets([
    ds_closed.repeat().shuffle(1000),
    ds_opened.repeat().shuffle(1000),
], [.5, .5]).shuffle(1000).batch(32).as_numpy_iterator()

In [21]:
batch = next(ds)

In [56]:
import flax.linen as nn
import jax
import jax.numpy as jnp
import optax
from flax.training import train_state


class MLP(nn.Module):

    @nn.compact
    def __call__(self, x):
        for i in range(4):
            x = jax.nn.swish(nn.Dense(256)(x))
        return nn.Dense(2)(x)


mlp = MLP()
params = mlp.init(jax.random.PRNGKey(0), batch[0])

state = train_state.TrainState.create(apply_fn=mlp.apply,
                                      params=params,
                                      tx=optax.adam(learning_rate=1e-4))

In [57]:
import pandas as pd


@jax.jit
def update_state(state: train_state.TrainState, batch):

    def loss_fn(params):
        pred = state.apply_fn(params, batch[0])
        loss = optax.softmax_cross_entropy_with_integer_labels(pred, batch[1])
        accuracy = jnp.mean(jnp.argmax(pred, axis=-1) == batch[1])
        return jnp.mean(loss), dict(acc=accuracy, loss=jnp.mean(loss))

    grads, metrics = jax.grad(loss_fn, has_aux=True)(state.params)

    return state.apply_gradients(grads=grads), metrics


l = []

for _ in range(200):
    state, m = update_state(state, next(ds))
    l.append(m)

In [61]:
from pythonosc import udp_client


udp = udp_client.SimpleUDPClient("localhost", 1234)

for landmarks in tracker.track():
    landmarks = landmarks[0].flatten()
    pred = state.apply_fn(state.params, landmarks[None])
    pred = jax.nn.softmax(pred.flatten())

    udp.send_message("/hand", [float(pred[0])])

E0000 00:00:1781604469.316349  226309 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-06-16T12:18:49.309198+02:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1781604529.321429  226309 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-06-16T12:18:49.309198+02:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
